# Aggregation Strategy 3: Cross-Attention (Task Queries → Views)

Learnable task-query tokens attend to the view features. Unlike attention pool, views are **combined through attention** rather than just re-weighted; unlike self-attention, the queries come from a **different source** than the keys/values (hence *cross*-attention).

## Definitional note

An earlier version of `CrossAttentionAggregate` in [`VARS model/mvaggregate.py`](../VARS%20model/mvaggregate.py) used `query = key = value = view_features`, which is self-attention with a residual — **not** cross-attention. The class was rewritten to the design below: Q from learnable task-query tokens, K/V from per-view features. That asymmetry is what makes it cross-attention.

The thesis Methods section must state this precisely: the word *cross* refers to the asymmetry between query and key/value sources, not to any inter-camera property.

## Formulation

Given per-view features $V \in \mathbb{R}^{B \times N \times d}$ and $K$ learnable task-query tokens $Q \in \mathbb{R}^{K \times d}$ (here $K = 2$, one per classification head):

1. Expand queries to batch: $Q_B = Q \in \mathbb{R}^{B \times K \times d}$.
2. Multi-head attention: $\tilde Q = \text{Attn}(\text{query}=Q_B, \text{key}=V, \text{value}=V)$.
3. LayerNorm: $\tilde Q = \text{LN}(\tilde Q) \in \mathbb{R}^{B \times K \times d}$.
4. Pool across queries: $z = \frac{1}{K} \sum_{k=1}^{K} \tilde Q_k$.

Each query token learns *what to look for* across the views. The attention weights $\alpha_{k,i}$ (query $k$ over view $i$) are interpretable: they say "for task $k$, how much does view $i$ matter?"

## Properties

| Property | Value |
|---|---|
| Learnable parameters | ≈ 1M ($K \cdot d$ queries + 4 MHA projections $d \times d$ + LN) |
| Views interact? | **Yes** — each query output mixes information across all views |
| Permutation-invariant over views | Yes |
| Asymmetric (Q source ≠ KV source)? | **Yes** — this is what distinguishes it from self-attention |
| Handles variable view count | Yes |

## What this isolates

Compared to attention pool, cross-attention adds *inter-view mixing* with *task conditioning*. If it beats attention pool significantly, the gain is attributable to one of those two mechanisms (and the thesis should discuss which is the likely driver — it's not possible to fully separate them without an explicit self-attention baseline, which is why adding it as a stretch-goal method would sharpen the comparison).

## Target implementation

This is the class that will replace the misdefined `CrossAttentionAggregate` in [`VARS model/mvaggregate.py`](../VARS%20model/mvaggregate.py) during Phase A. Defined inline here so the notebook is runnable today; once Phase A is merged, this cell can be replaced with `from mvaggregate import CrossAttentionAggregate`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../VARS model'))

import torch
from torch import nn
from utils import batch_tensor, unbatch_tensor

class CrossAttentionAggregate(nn.Module):
    """Task-conditioned cross-attention over camera views.

    Q comes from learnable task-query tokens; K/V come from per-view features.
    """
    def __init__(self, model, feat_dim, num_heads=8, num_queries=2, lifting_net=nn.Sequential()):
        super().__init__()
        self.model = model
        self.lifting_net = lifting_net
        self.feat_dim = feat_dim
        self.num_queries = num_queries

        self.query_tokens = nn.Parameter(torch.randn(num_queries, feat_dim) * 0.02)
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=feat_dim, num_heads=num_heads,
            dropout=0.1, batch_first=True,
        )
        self.norm = nn.LayerNorm(feat_dim)

    def forward(self, mvimages):
        B, V, C, D, H, W = mvimages.shape
        aux = self.lifting_net(unbatch_tensor(
            self.model(batch_tensor(mvimages, dim=1, squeeze=True)),
            B, dim=1, unsqueeze=True,
        ))  # (B, V, feat_dim)

        q = self.query_tokens.unsqueeze(0).expand(B, -1, -1)  # (B, K, d)
        attended, attn = self.cross_attention(
            query=q, key=aux, value=aux,
            need_weights=True, average_attn_weights=True,
        )
        attended = self.norm(attended)  # (B, K, d)
        pooled = attended.mean(dim=1)   # (B, d)
        return pooled.squeeze(), attn

print('CrossAttentionAggregate defined inline (target design for Phase A).')

/venv/vars/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CrossAttentionAggregate defined inline (target design for Phase A).


## Forward-pass demo on the inner aggregation logic

Again bypassing the backbone to show just the cross-attention math.

In [2]:
torch.manual_seed(0)
B, V, d, K = 2, 3, 512, 2  # K = num_queries (one per task head)

view_features = torch.randn(B, V, d)
query_tokens = nn.Parameter(torch.randn(K, d) * 0.02)
mha = nn.MultiheadAttention(embed_dim=d, num_heads=8, batch_first=True)
ln = nn.LayerNorm(d)

q = query_tokens.unsqueeze(0).expand(B, -1, -1)
attended, attn = mha(query=q, key=view_features, value=view_features,
                     need_weights=True, average_attn_weights=True)
attended = ln(attended)
pooled = attended.mean(dim=1)

print('Per-view features:  ', view_features.shape)
print('Query tokens:       ', query_tokens.shape, f'({K} learnable task queries)')
print('Attention weights:  ', attn.shape, '(B, K queries, V views)')
print('Pooled feature:     ', pooled.shape)

Per-view features:   torch.Size([2, 3, 512])
Query tokens:        torch.Size([2, 512]) (2 learnable task queries)
Attention weights:   torch.Size([2, 2, 3]) (B, K queries, V views)
Pooled feature:      torch.Size([2, 512])


### Interpretability bonus

`attn[b, k, i]` tells you how much task-query $k$ attended to view $i$ for sample $b$. After training, this gives you per-class view importance — a figure that will carry weight in the thesis discussion.

In [3]:
print('Attention weights for sample 0:')
print(f'  Query 0 (offence/severity) over {V} views: {attn[0, 0].tolist()}')
print(f'  Query 1 (action type)      over {V} views: {attn[0, 1].tolist()}')

# After training, these weights are interpretable: a high α_{0, i} means
# "the model leans on view i when deciding offence/severity for this sample."

Attention weights for sample 0:
  Query 0 (offence/severity) over 3 views: [0.33356043696403503, 0.3332093060016632, 0.33323028683662415]
  Query 1 (action type)      over 3 views: [0.3318396508693695, 0.33417460322380066, 0.33398574590682983]


In [4]:
# Parameter count of the aggregation block alone
class IdentityBackbone(nn.Module):
    def forward(self, x):
        return x

agg = CrossAttentionAggregate(model=IdentityBackbone(), feat_dim=512)
own_params = sum(p.numel() for name, p in agg.named_parameters() if not name.startswith('model.'))
print(f'Cross-attention aggregation block parameters: {own_params:,}')

Cross-attention aggregation block parameters: 1,052,672
